# Various model pipelines

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use project root relative to this notebook
BASE_DIR = Path('/home/amraas/projects/realestatecons')
DATA_DIR = BASE_DIR / 'data' / 'raw'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

In [2]:
import sys

sys.path.append(str(BASE_DIR))

from src.data.preprocess import clean_test_data, clean_train_data
from src.features.features import (
    add_engineered_features,
    add_log_transformed_features,
    drop_highly_correlated_features,
)

In [3]:
train_df_cleaned = clean_train_data(train_df)
test_df_cleaned = clean_test_data(test_df, train_df)

train_df_features = add_engineered_features(train_df_cleaned)
test_df_features = add_engineered_features(test_df_cleaned)

In [4]:
train_df = drop_highly_correlated_features(train_df_features)
test_df = drop_highly_correlated_features(test_df_features)

In [5]:
train_df.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearRemodAdd', 'RoofStyle', 'RoofMatl',
       'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
       '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'BsmtFullBath', 'BsmtHalfBath',
       'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageQual', 'GarageCond',
       'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3Ss

## Splitting

In [6]:
train_df.shape

(1460, 83)

In [7]:
test_df.shape

(1459, 82)

In [8]:
from sklearn.model_selection import train_test_split
X_train = train_df.drop(columns='SalePrice')
y_train = train_df['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)

In [9]:
X_train.shape, X_val.shape

((1022, 82), (438, 82))

## Linear Regression

In [10]:
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler

SUBMISSIONS_DIR = BASE_DIR / "reports" / "figures" / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

X_full = train_df.drop(columns="SalePrice")
y_full = train_df["SalePrice"]

ORDINAL_CATEGORIES = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "FireplaceQu": ["Po", "Fa", "TA", "Gd", "Ex"],
    "GarageQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "PoolQC": ["Fa", "TA", "Gd", "Ex"],
    "BsmtExposure": ["No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "GarageFinish": ["Unf", "RFn", "Fin"],
    "Fence": ["MnWw", "GdWo", "MnPrv", "GdPrv"],
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "PavedDrive": ["N", "P", "Y"],
}


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features]

    ordinal_features = [c for c in categorical_features if c in ORDINAL_CATEGORIES]
    nominal_features = [c for c in categorical_features if c not in ORDINAL_CATEGORIES]

    numeric_transformer = Pipeline(
        steps=[("scaler", RobustScaler())]
    )
    ordinal_transformer = OrdinalEncoder(
        categories=[ORDINAL_CATEGORIES[c] for c in ordinal_features],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )
    nominal_transformer = OneHotEncoder(handle_unknown="ignore")

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("ord", ordinal_transformer, ordinal_features),
            ("nom", nominal_transformer, nominal_features),
        ]
    )


def save_submission(preds: np.ndarray, name: str) -> Path:
    submission = pd.read_csv(BASE_DIR / "sample_submission.csv")
    submission["SalePrice"] = preds
    out_path = SUBMISSIONS_DIR / f"{name}.csv"
    submission.to_csv(out_path, index=False)
    print(f"Saved submission: {out_path}")
    return out_path


def run_grid_search(
    model,
    param_grid: dict,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_name: str,
) -> Pipeline:
    preprocessor = build_preprocessor(X_train)
    pipeline = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)

    scoring = {
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    }
    cv_results = cross_validate(
        grid.best_estimator_, X_train, y_train, cv=cv, scoring=scoring
    )

    print(f"{model_name} best params: {grid.best_params_}")
    print(
        f"{model_name} CV mean RMSE: {-cv_results['test_rmse'].mean():.4f}"
    )
    print(
        f"{model_name} CV mean MAE: {-cv_results['test_mae'].mean():.4f}"
    )
    print(f"{model_name} CV mean R2: {cv_results['test_r2'].mean():.4f}")

    return grid.best_estimator_


# Linear Regression via SGDRegressor
linear_model = SGDRegressor(
    penalty=None, random_state=42, learning_rate="invscaling"
)
linear_param_grid = {
    "model__eta0": [0.001, 0.01, 0.1],
    "model__max_iter": [1000, 2000],
}

best_linear = run_grid_search(
    linear_model, linear_param_grid, X_train, y_train, "LinearRegression"
)
best_linear.fit(X_full, y_full)
linear_preds = best_linear.predict(test_df)
save_submission(linear_preds, "linear_regression")

LinearRegression best params: {'model__eta0': 0.001, 'model__max_iter': 1000}
LinearRegression CV mean RMSE: 25470605705214.4180
LinearRegression CV mean MAE: 5485176614002.2754
LinearRegression CV mean R2: -176673275535164128.0000
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/linear_regression.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/linear_regression.csv')

## Lasso Regression

In [11]:
# Lasso Regression via SGDRegressor
lasso_model = SGDRegressor(
    penalty="l1", random_state=42, learning_rate="invscaling"
)
lasso_param_grid = {
    "model__eta0": [0.001, 0.01, 0.1],
    "model__max_iter": [1000, 2000,10000],
    "model__alpha": [1e-4, 1e-3, 1e-2, 1, 5],
}

best_lasso = run_grid_search(
    lasso_model, lasso_param_grid, X_train, y_train, "Lasso"
)
best_lasso.fit(X_full, y_full)
lasso_preds = best_lasso.predict(test_df)
save_submission(lasso_preds, "lasso")

Lasso best params: {'model__alpha': 0.001, 'model__eta0': 0.001, 'model__max_iter': 1000}
Lasso CV mean RMSE: 23574258434135.5312
Lasso CV mean MAE: 5441282272474.3418
Lasso CV mean R2: -163397159791676992.0000
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/lasso.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/lasso.csv')

## Ridge Regression

In [12]:
# Ridge Regression via SGDRegressor
ridge_model = SGDRegressor(
    penalty="l2", random_state=42, learning_rate="invscaling"
)
ridge_param_grid = {
    "model__eta0": [0.001, 0.01, 0.1],
    "model__max_iter": [1000, 2000],
    "model__alpha": [1e-4, 1e-3, 1e-2, 1, 5, 10],
}

best_ridge = run_grid_search(
    ridge_model, ridge_param_grid, X_train, y_train, "Ridge"
)
best_ridge.fit(X_full, y_full)
ridge_preds = best_ridge.predict(test_df)
save_submission(ridge_preds, "ridge")

Ridge best params: {'model__alpha': 5, 'model__eta0': 0.001, 'model__max_iter': 1000}
Ridge CV mean RMSE: 22814832387971.2188
Ridge CV mean MAE: 4716510127579.9980
Ridge CV mean R2: -164684684400780064.0000
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/ridge.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/ridge.csv')

## Random Forest

In [13]:
# Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42)
rf_param_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"],
}

best_rf = run_grid_search(
    rf_model, rf_param_grid, X_train, y_train, "RandomForest"
)
best_rf.fit(X_full, y_full)
rf_preds = best_rf.predict(test_df)
save_submission(rf_preds, "random_forest")

RandomForest best params: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__n_estimators': 500}
RandomForest CV mean RMSE: 31725.5897
RandomForest CV mean MAE: 17932.3307
RandomForest CV mean R2: 0.8289
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/random_forest.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/random_forest.csv')

## Gradient Boosting (XGBoost)

In [14]:
try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError("xgboost is required for this section. Install it first.") from exc

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    tree_method="hist",
)

xgb_param_grid = {
    "model__n_estimators": [300, 600],
    "model__learning_rate": [0.03, 0.1],
    "model__max_depth": [3, 6],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}

best_xgb = run_grid_search(
    xgb_model, xgb_param_grid, X_train, y_train, "XGBoost"
)
best_xgb.fit(X_full, y_full)
xgb_preds = best_xgb.predict(test_df)
save_submission(xgb_preds, "xgboost")

XGBoost best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 3, 'model__n_estimators': 600, 'model__subsample': 0.8}
XGBoost CV mean RMSE: 28414.5641
XGBoost CV mean MAE: 15607.9975
XGBoost CV mean R2: 0.8594
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/xgboost.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/xgboost.csv')

## Log transform effect on linear models

In [17]:
log_transform_cols = ["OverallQual", "YearBuilt", "Age", "MSSubClass", "SalePrice"]

train_df_log = add_log_transformed_features(train_df, log_transform_cols)
test_df_log = add_log_transformed_features(test_df, log_transform_cols)

X_full_log = train_df_log.drop(columns="SalePrice")
y_full_log = train_df_log["SalePrice"]

X_train_log = X_full_log.copy()
y_train_log = y_full_log.copy()

# Linear Regression (log features)
best_linear_log = run_grid_search(
    linear_model, linear_param_grid, X_train_log, y_train_log, "LinearRegression_Log"
)
best_linear_log.fit(X_full_log, y_full_log)
# linear_log_preds = best_linear_log.predict(test_df_log)
# save_submission(linear_log_preds, "linear_regression_log")

# Lasso (log features)
best_lasso_log = run_grid_search(
    lasso_model, lasso_param_grid, X_train_log, y_train_log, "Lasso_Log"
)
best_lasso_log.fit(X_full_log, y_full_log)
# lasso_log_preds = best_lasso_log.predict(test_df_log)
# save_submission(lasso_log_preds, "lasso_log")

# Ridge (log features)
best_ridge_log = run_grid_search(
    ridge_model, ridge_param_grid, X_train_log, y_train_log, "Ridge_Log"
)
best_ridge_log.fit(X_full_log, y_full_log)
# ridge_log_preds = best_ridge_log.predict(test_df_log)
# save_submission(ridge_log_preds, "ridge_log")

/home/amraas/miniconda3/envs/realestateconsultant/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


LinearRegression_Log best params: {'model__eta0': 0.001, 'model__max_iter': 1000}
LinearRegression_Log CV mean RMSE: 130827644513460.0938
LinearRegression_Log CV mean MAE: 16752036573658.9648
LinearRegression_Log CV mean R2: -6038751962429476864.0000
Lasso_Log best params: {'model__alpha': 5, 'model__eta0': 0.001, 'model__max_iter': 1000}
Lasso_Log CV mean RMSE: 120850684130903.2812
Lasso_Log CV mean MAE: 15479232199824.3164
Lasso_Log CV mean R2: -5571548011156277248.0000
Ridge_Log best params: {'model__alpha': 10, 'model__eta0': 0.001, 'model__max_iter': 1000}
Ridge_Log CV mean RMSE: 123882236800010.0000
Ridge_Log CV mean MAE: 16108794610885.6152
Ridge_Log CV mean R2: -4542402491100544000.0000


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

## Effect of log transform + dim reduction on Linear models

In [25]:
from sklearn.decomposition import PCA

log_transform_cols = ["OverallQual", "YearBuilt", "Age", "MSSubClass", "SalePrice"]

train_df_log = add_log_transformed_features(train_df, log_transform_cols)
test_df_log = add_log_transformed_features(test_df, log_transform_cols)

X_full_log = train_df_log.drop(columns="SalePrice")
y_full_log = train_df_log["SalePrice"]

pca_preprocessor = Pipeline(
    steps=[
        ("preprocess", build_preprocessor(X_full_log)),
        ("pca", PCA(n_components=5, random_state=42)),
    ]
)

X_full_pca = pca_preprocessor.fit_transform(X_full_log)

# Linear Regression (log + PCA)
linear_pca = SGDRegressor(
    penalty=None, random_state=42, learning_rate="invscaling"
)
linear_pca_param_grid = {
    "eta0": [0.001, 0.01, 0.1],
    "max_iter": [1000, 2000],
}

linear_pca_grid = GridSearchCV(
    linear_pca,
    param_grid=linear_pca_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
)
linear_pca_grid.fit(X_full_pca, y_full_log)

print(f"LinearRegression_LogPCA best params: {linear_pca_grid.best_params_}")

# Lasso (log + PCA)
lasso_pca = SGDRegressor(
    penalty="l1", random_state=42, learning_rate="invscaling"
)
lasso_pca_param_grid = {
    "eta0": [0.001, 0.01, 0.1],
    "max_iter": [1000, 2000],
    "alpha": [1e-4, 1e-3, 1e-2],
}

lasso_pca_grid = GridSearchCV(
    lasso_pca,
    param_grid=lasso_pca_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
)
lasso_pca_grid.fit(X_full_pca, y_full_log)

print(f"Lasso_LogPCA best params: {lasso_pca_grid.best_params_}")

# Ridge (log + PCA)
ridge_pca = SGDRegressor(
    penalty="l2", random_state=42, learning_rate="invscaling"
)
ridge_pca_param_grid = {
    "eta0": [0.001, 0.01, 0.1],
    "max_iter": [1000, 2000],
    "alpha": [1e-4, 1e-3, 1e-2],
}

ridge_pca_grid = GridSearchCV(
    ridge_pca,
    param_grid=ridge_pca_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
)
ridge_pca_grid.fit(X_full_pca, y_full_log)

print(f"Ridge_LogPCA best params: {ridge_pca_grid.best_params_}")

/home/amraas/miniconda3/envs/realestateconsultant/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


LinearRegression_LogPCA best params: {'eta0': 0.001, 'max_iter': 1000}
Lasso_LogPCA best params: {'alpha': 0.01, 'eta0': 0.001, 'max_iter': 1000}
Ridge_LogPCA best params: {'alpha': 0.001, 'eta0': 0.001, 'max_iter': 1000}


In [26]:
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("LinearRegression_LogPCA metrics:")
linear_pca_cv = cross_validate(
    linear_pca_grid.best_estimator_, X_full_pca, y_full_log, cv=cv, scoring=scoring
)
print(f"RMSE: {-linear_pca_cv['test_rmse'].mean():.4f}")
print(f"MAE: {-linear_pca_cv['test_mae'].mean():.4f}")
print(f"R2: {linear_pca_cv['test_r2'].mean():.4f}")

print("\nLasso_LogPCA metrics:")
lasso_pca_cv = cross_validate(
    lasso_pca_grid.best_estimator_, X_full_pca, y_full_log, cv=cv, scoring=scoring
)
print(f"RMSE: {-lasso_pca_cv['test_rmse'].mean():.4f}")
print(f"MAE: {-lasso_pca_cv['test_mae'].mean():.4f}")
print(f"R2: {lasso_pca_cv['test_r2'].mean():.4f}")

print("\nRidge_LogPCA metrics:")
ridge_pca_cv = cross_validate(
    ridge_pca_grid.best_estimator_, X_full_pca, y_full_log, cv=cv, scoring=scoring
)
print(f"RMSE: {-ridge_pca_cv['test_rmse'].mean():.4f}")
print(f"MAE: {-ridge_pca_cv['test_mae'].mean():.4f}")
print(f"R2: {ridge_pca_cv['test_r2'].mean():.4f}")

LinearRegression_LogPCA metrics:
RMSE: 14307197688577.6660
MAE: 3410663253967.9497
R2: -65002312375438768.0000

Lasso_LogPCA metrics:
RMSE: 15256327866910.5977
MAE: 3610778798104.6768
R2: -69616454358775104.0000

Ridge_LogPCA metrics:
RMSE: 11824418709382.8477
MAE: 3107077685239.3472
R2: -34662764444984248.0000
